# 22 - USS region vs image-only priors on UNSEEN obstacle types

Notebook 21 showed SAM3 text prompt "object" inside the USS region finds the
obstacle in 10/11 detector-missed sequences (all cones) - but it lacked a fair
no-USS baseline and covered one type. Here, one SAM3 "object" call per frame,
four ways to pick the mask:

| variant | USS? | pick |
|---|---|---|
| uss_region | yes | best score with centre inside the USS region |
| uss_nearest | yes | box bottom nearest the projected USS point (inside region) |
| nouss_best | no | best score anywhere |
| nouss_corridor | no | score x approach-corridor prior (our labeling pipeline's image-only prior) |

Types: cone, bush, bollard, car (each unseen to its held-out detector model:
baseline / bush / bollard rotations / no-bucket) + pole, curbstone as seen
controls. No training happens here (SAM3 zero-shot + calibration), so any
split's sequences can be evaluated. Calibration is CROSS-FIT within each
(camera, day): sequences alternate into two folds, each evaluated with the
camera fitted on the other fold - no sequence's own label touches its camera.
GT = pseudo-label box; degenerate far boxes (< 15 px tall / 8 px wide) dropped.
GPU required.

In [ ]:
# --- pipeline bootstrap (identical first cell in every pipeline notebook) ---
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/vision-uss-research")
    REPO_URL = "https://github.com/Ali-RT/vision-uss-research"
    if not (REPO_DIR / ".git").exists():   # fresh VM: clone (no separate bootstrap script)
        subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
    # always run the latest main; discards VM-local edits to tracked repo files
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "-q", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "-q", "--hard", "origin/main"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(REPO_DIR / "requirements" / "colab.txt")], check=True)
    # canmatrix 1.3.0 (2026-10-01) breaks `import asammdf` (TypeError in
    # asammdf/blocks/types.py) - pin until asammdf supports it
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "asammdf==8.8.27",
                    "canmatrix==1.2"], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "pyproject.toml").exists())

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from vision_uss_research.runs import git_sha
from vision_uss_research.settings import load_paths

PROFILE = "colab_drive" if IN_COLAB else "local"
ARTIFACTS_OVERRIDE = None
RAW_ROOT_OVERRIDE = None

paths = load_paths(profile=PROFILE)
ARTIFACTS = Path(ARTIFACTS_OVERRIDE or paths.outputs_dir)
RAW_ROOT = Path(RAW_ROOT_OVERRIDE or paths.raw_data_root)
print(f"repo      : {REPO_DIR}")
print(f"git sha   : {git_sha(REPO_DIR)}")
print(f"artifacts : {ARTIFACTS}")
print(f"raw root  : {RAW_ROOT}")

In [ ]:
import json as _json
import sys

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

sys.path.insert(0, str(REPO_DIR / "scripts"))
import uss_prompt_analysis as upa
from vision_uss_research.labeling.corridor import pick_target_instance
from vision_uss_research.method.uss_prompt import (fit_ground_camera, region_box,
                                                   track_target_slots)
from vision_uss_research.qa.flags import box_iou
from vision_uss_research.runs import finish_run, new_run_id, start_run
from vision_uss_research.sequences import frames_window_id

DATASET_RUN = "20260718_dataset_v2"
TYPES_RUN = "dataset_v2daysplit"          # manifest carrying target_object per sequence
PROJ_RUN = "prompt_uss_proj_v1"
TYPES = ["cone", "bush", "bollard", "car", "pole", "curbstone"]
UNSEEN = {"cone", "bush", "bollard", "car"}
SEQS_PER_TYPE = 60
FRAMES_PER_SEQ = 5
REGION = dict(half_w=160, below=100, above=250)
PROMPT = "object"
SCORE_THRESHOLD = 0.2
FOUND_IOU = 0.3
MIN_GT_H, MIN_GT_W = 15, 8
SEED = 0

MAN_DIR = ARTIFACTS / "datasets" / DATASET_RUN
PROJ_DIR = ARTIFACTS / "method" / PROJ_RUN
WINDOW_ROOT = ARTIFACTS / "frames" / frames_window_id(20, (0.30, 0.85))
RUN_DIR = ARTIFACTS / "method" / new_run_id("prompt", "unseen_types_v1")
OUT_CSV = RUN_DIR / "unseen_types.csv"
RUN_DIR.mkdir(parents=True, exist_ok=True)
KEYS = ["sequence_id", "camera", "frame_idx"]

slots, boxes = upa.load(PROJ_DIR, MAN_DIR)
types = (pd.read_csv(ARTIFACTS / "datasets" / TYPES_RUN / "dataset_manifest.csv",
                     usecols=["sequence_id", "target_object"]).drop_duplicates("sequence_id"))
pairs = (slots.merge(track_target_slots(slots), on=["sequence_id", "camera", "slot"])
         .merge(boxes, on=KEYS).merge(types, on="sequence_id"))
pairs["day"] = pairs.sequence_id.str.split("_").str[1]
pairs = pairs[((pairs.y1 - pairs.y0) >= MIN_GT_H) & ((pairs.x1 - pairs.x0) >= MIN_GT_W)]

# cross-fit calibration: two folds per (camera, day), global per fold as fallback
seq_fold = {}
for (_, _), d in pairs.groupby(["camera", "day"]):
    for i, s in enumerate(sorted(d.sequence_id.unique())):
        seq_fold[s] = i % 2
pairs["fold"] = pairs.sequence_id.map(seq_fold)
def fit(d, cam_name):
    return fit_ground_camera(d.mid_x, d.mid_y, (d.x0 + d.x1) / 2, d.y1, cam_name)[0]
glob = {(c, f): fit(pairs[(pairs.camera == c) & (pairs.fold == f)], c)
        for c in ("front", "rear") for f in (0, 1)}
cams = {}
for (c, day, f), d in pairs.groupby(["camera", "day", "fold"]):
    if len(d) >= 25:
        cams[(c, day, f)] = fit(d, c)
print(f"per-day-fold calibrations: {len(cams)}")

rng = np.random.default_rng(SEED)
chosen = []
for t in TYPES:
    seqs = pairs[pairs.target_object == t].sequence_id.drop_duplicates()
    seqs = seqs.sample(min(SEQS_PER_TYPE, len(seqs)), random_state=SEED)
    sub = pairs[pairs.sequence_id.isin(seqs)].sample(frac=1, random_state=SEED)
    chosen.append(sub.groupby("sequence_id").head(FRAMES_PER_SEQ))
todo = pd.concat(chosen, ignore_index=True)
proj = []
for r in todo.itertuples(index=False):
    other = 1 - r.fold
    m = cams.get((r.camera, r.day, other))
    src = "day"
    if m is None:
        m, src = glob[(r.camera, other)], "global"
    u, v, ok = m.project(r.mid_x, r.mid_y)
    proj.append((float(u), float(v), bool(ok), src))
todo["pu"], todo["pv"], todo["proj_ok"], todo["calib"] = zip(*proj)
todo = todo[todo.proj_ok]

done = set()
if OUT_CSV.exists():
    done = set(map(tuple, pd.read_csv(OUT_CSV)[KEYS].astype(str).values))
todo = todo[[tuple(map(str, k)) not in done for k in todo[KEYS].values]]
start_run(RUN_DIR, config={"types": TYPES, "seqs_per_type": SEQS_PER_TYPE, "prompt": PROMPT,
                           "region": REGION, "found_iou": FOUND_IOU},
          inputs=[f"method/{PROJ_RUN}", f"datasets/{DATASET_RUN}", f"datasets/{TYPES_RUN}"],
          repo_root=REPO_DIR)
print(todo.groupby("target_object").agg(seqs=("sequence_id", "nunique"),
                                         frames=("frame_idx", "size")).to_string())
print(f"calibration: {todo.calib.value_counts().to_dict()} | todo frames {len(todo)}")

## SAM3 'object' once per frame, four picks

In [ ]:
from vision_uss_research.labeling.sam3 import Sam3Engine
eng = Sam3Engine(score_threshold=SCORE_THRESHOLD)

def inside(b, reg):
    cx, cy = (b[0] + b[2]) / 2, (b[1] + b[3]) / 2
    return reg[0] <= cx <= reg[2] and reg[1] <= cy <= reg[3]

buf = []
def flush():
    global buf
    if buf:
        pd.DataFrame(buf).to_csv(OUT_CSV, mode="a", header=not OUT_CSV.exists(), index=False)
        buf = []

for r in tqdm(todo.itertuples(index=False), total=len(todo), unit="frame"):
    out = {"sequence_id": r.sequence_id, "camera": r.camera, "frame_idx": r.frame_idx,
           "target_object": r.target_object, "unseen": int(r.target_object in UNSEEN),
           "range_mm": r.range_mm, "calib": r.calib, "pu": round(r.pu, 1),
           "pv": round(r.pv, 1), "error": ""}
    p = WINDOW_ROOT / r.sequence_id / r.camera / f"{int(r.frame_idx):05d}.jpg"
    if not p.exists():
        out["error"] = "no_frame"; buf.append(out); continue
    img = np.array(Image.open(p).convert("RGB"))
    H, W = img.shape[:2]
    gt = (r.x0, r.y0, r.x1, r.y1)
    reg = region_box(r.pu, r.pv, W, H, **REGION)
    out.update(gt_in_region=int(inside(gt, reg)), x0=r.x0, y0=r.y0, x1=r.x1, y1=r.y1)
    try:
        res = eng.segment_text(img, PROMPT)
    except Exception as e:
        out["error"] = f"text:{type(e).__name__}"; buf.append(out); continue
    cand = [(tuple(map(float, b)), float(s)) for b, s in zip(res["boxes"], res["scores"])]
    out["n_candidates"] = len(cand)
    picks = {}
    inr = [c for c in cand if inside(c[0], reg)]
    picks["uss_region"] = max(inr, key=lambda c: c[1]) if inr else None
    picks["uss_nearest"] = (min(inr, key=lambda c: np.hypot((c[0][0] + c[0][2]) / 2 - r.pu,
                                                            c[0][3] - r.pv)) if inr else None)
    picks["nouss_best"] = max(cand, key=lambda c: c[1]) if cand else None
    if cand:
        i, _ = pick_target_instance({"boxes": np.array([c[0] for c in cand]),
                                     "scores": np.array([c[1] for c in cand])}, (H, W))
        picks["nouss_corridor"] = cand[i] if i is not None else None
    else:
        picks["nouss_corridor"] = None
    for k, c in picks.items():
        out[f"{k}_iou"] = round(box_iou(c[0], gt), 4) if c else 0.0
        out[f"{k}_box"] = None if c is None else ",".join(f"{v:.0f}" for v in c[0])
    buf.append(out)
    if len(buf) >= 50:
        flush()
    eng.free()
flush()

## Summary

In [ ]:
res = pd.read_csv(OUT_CSV)
ok = res[res.error.fillna("") == ""]
V = ["uss_region", "uss_nearest", "nouss_best", "nouss_corridor"]
rows = []
for t, d in ok.groupby("target_object"):
    row = {"type": t, "unseen": int(t in UNSEEN), "seqs": d.sequence_id.nunique(),
           "frames": len(d), "gt_in_region": round(d.gt_in_region.mean(), 3)}
    for v in V:
        f = d[f"{v}_iou"] >= FOUND_IOU
        row[f"{v}_frame"] = round(f.mean(), 3)
        row[f"{v}_seq"] = round(d.assign(f=f).groupby("sequence_id").f.any().mean(), 3)
    rows.append(row)
tab = pd.DataFrame(rows).sort_values(["unseen", "type"], ascending=[False, True])
pd.set_option("display.width", 250); display(tab)
tab.to_csv(RUN_DIR / "summary.csv", index=False)

# paired comparison on unseen types: USS-region vs the image-only corridor prior
u = ok[ok.unseen == 1]
a = u.uss_region_iou >= FOUND_IOU; b = u.nouss_corridor_iou >= FOUND_IOU
print(f"\nUNSEEN frames {len(u)}: uss_region {a.mean():.3f} vs nouss_corridor {b.mean():.3f} | "
      f"USS-only wins {int((a & ~b).sum())}, corridor-only wins {int((~a & b).sum())}")
finish_run(RUN_DIR, summary={"rows": tab.to_dict(orient="records"),
                             "unseen_frames": int(len(u)),
                             "uss_region_frame": round(float(a.mean()), 4),
                             "nouss_corridor_frame": round(float(b.mean()), 4)})

# gallery: bush + bollard frames - GT green, region yellow, USS pick red, corridor pick blue
import matplotlib.pyplot as plt
g = ok[ok.target_object.isin(["bush", "bollard"])].sample(min(12, len(ok)), random_state=SEED)
fig, axes = plt.subplots(3, 4, figsize=(16, 8))
for ax, r in zip(axes.flat, g.itertuples()):
    im = cv2.cvtColor(cv2.imread(str(WINDOW_ROOT / r.sequence_id / r.camera /
                                     f"{int(r.frame_idx):05d}.jpg")), cv2.COLOR_BGR2RGB)
    H, W = im.shape[:2]
    reg = region_box(r.pu, r.pv, W, H, **REGION)
    cv2.rectangle(im, (int(reg[0]), int(reg[1])), (int(reg[2]), int(reg[3])), (255, 220, 0), 2)
    cv2.rectangle(im, (int(r.x0), int(r.y0)), (int(r.x1), int(r.y1)), (0, 255, 0), 2)
    for col, color in (("uss_region_box", (255, 0, 0)), ("nouss_corridor_box", (0, 120, 255))):
        val = getattr(r, col)
        if isinstance(val, str):
            x0, y0, x1, y1 = map(float, val.split(","))
            cv2.rectangle(im, (int(x0), int(y0)), (int(x1), int(y1)), color, 2)
    ax.imshow(im); ax.axis("off")
    ax.set_title(f"{r.target_object} USS {r.uss_region_iou:.2f} / corr {r.nouss_corridor_iou:.2f}",
                 fontsize=8)
fig.tight_layout(); fig.savefig(RUN_DIR / "gallery_unseen.jpg", dpi=90); plt.show()
print(f"download {RUN_DIR.name}/ and share summary.csv + gallery_unseen.jpg")